# AGGRESSIVE

# LSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000  # or use len(tokenizer.word_index) + 1 later
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights if needed (optional, helps with imbalance)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # sigmoid for binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=4,
    class_weight=class_weights  # Optional, based on class balance
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 41s 57ms/step - accuracy: 0.5068 - loss: 0.6975 - val_accuracy: 0.5000 - val_loss: 0.6929
Epoch 2/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 36s 54ms/step - accuracy: 0.5044 - loss: 0.6944 - val_accuracy: 0.6426 - val_loss: 0.6642
Epoch 3/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 41s 54ms/step - accuracy: 0.4973 - loss: 0.7118 - val_accuracy: 0.5080 - val_loss: 0.6924
Epoch 4/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 39s 51ms/step - accuracy: 0.5064 - loss: 0.6883 - val_accuracy: 0.5096 - val_loss: 0.6891
Epoch 5/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 42s 52ms/step - accuracy: 0.5027 - loss: 0.6850 - val_accuracy: 0.5080 - val_loss: 0.6920
Epoch 6/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 41s 52ms/step - accuracy: 0.4917 - loss: 0.6863 - val_accuracy: 0.5208 - val_loss: 0.6827
Epoch 7/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 42s 54ms/step - accuracy: 0.5627 - loss: 0.6716 - val_accuracy: 0.5080 - val_loss: 0.6933
Epoch 8/20
675/675 ━━━━━━━━━━━━━━━━━━━━ 43s 56ms/step - accuracy: 0.5738 - loss: 0.6662 - val_accurac

# BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights (optional but useful for imbalance)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build BiLSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 63s 85ms/step - accuracy: 0.7089 - loss: 0.5525 - val_accuracy: 0.8446 - val_loss: 0.3363
Epoch 2/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 85ms/step - accuracy: 0.9458 - loss: 0.1456 - val_accuracy: 0.8446 - val_loss: 0.3998
Epoch 3/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 84ms/step - accuracy: 0.9841 - loss: 0.0483 - val_accuracy: 0.8317 - val_loss: 0.6022
Epoch 4/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 85s 89ms/step - accuracy: 0.9980 - loss: 0.0161 - val_accuracy: 0.8317 - val_loss: 0.8139
Epoch 5/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 79s 84ms/step - accuracy: 0.9999 - loss: 0.0041 - val_accuracy: 0.8478 - val_loss: 0.7585
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.8313    0.8846    0.8571       312
           1     0.8771    0.8211    0.8482       313

    accuracy                         0.8528       625
   macro avg     0.8542    0.8529    0.8527       625
weighted avg     0.8543    0

# LSTM+BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),  # First LSTM layer (return sequences so next LSTM can take it)
    Bidirectional(LSTM(64, return_sequences=False)),  # BiLSTM on top
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=4,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/4


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 83s 113ms/step - accuracy: 0.6899 - loss: 0.5887 - val_accuracy: 0.8413 - val_loss: 0.3652
Epoch 2/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 80s 110ms/step - accuracy: 0.9413 - loss: 0.1788 - val_accuracy: 0.8365 - val_loss: 0.4364
Epoch 3/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 85s 114ms/step - accuracy: 0.9847 - loss: 0.0569 - val_accuracy: 0.7740 - val_loss: 0.4949
Epoch 4/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 115ms/step - accuracy: 0.9884 - loss: 0.0429 - val_accuracy: 0.8189 - val_loss: 0.8244
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.7559    0.9231    0.8312       312
           1     0.9016    0.7029    0.7899       313

    accuracy                         0.8128       625
   macro avg     0.8288    0.8130    0.8106       625
weighted avg     0.8289    0.8128    0.8105       625



# LSTM+BILSTM+CNN

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM + CNN model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),
    Bidirectional(LSTM(64, return_sequences=True)),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/3


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 86s 117ms/step - accuracy: 0.6485 - loss: 0.6265 - val_accuracy: 0.7949 - val_loss: 0.4449
Epoch 2/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 80s 115ms/step - accuracy: 0.9298 - loss: 0.2296 - val_accuracy: 0.8574 - val_loss: 0.3427
Epoch 3/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 84s 118ms/step - accuracy: 0.9734 - loss: 0.0707 - val_accuracy: 0.8558 - val_loss: 0.5986
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.8431    0.8782    0.8603       312
           1     0.8733    0.8371    0.8548       313

    accuracy                         0.8576       625
   macro avg     0.8582    0.8576    0.8575       625
weighted avg     0.8582    0.8576    0.8575       625



# SENTIMENT

# LSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000  # or use len(tokenizer.word_index) + 1 later
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights if needed (optional, helps with imbalance)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')  # sigmoid for binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights  # Optional, based on class balance
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.5327 - loss: 0.6943 - val_accuracy: 0.5016 - val_loss: 0.6938
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.5090 - loss: 0.6932 - val_accuracy: 0.5016 - val_loss: 0.6932
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.4783 - loss: 0.6935 - val_accuracy: 0.5016 - val_loss: 0.6931
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.4920 - loss: 0.6936 - val_accuracy: 0.5048 - val_loss: 0.6929
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.4885 - loss: 0.6934 - val_accuracy: 0.5000 - val_loss: 0.6930
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.5049 - loss: 0.6944 - val_accuracy: 0.5000 - val_loss: 0.6915
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.4985 - loss: 0.6924 - val_accuracy: 0.5032 - val_loss: 0.6928
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.5078 - loss: 0.6921 - val_accuracy: 0.50

# BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")


# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights (optional but useful for imbalance)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build BiLSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 64s 88ms/step - accuracy: 0.5139 - loss: 0.6888 - val_accuracy: 0.6763 - val_loss: 0.6155
Epoch 2/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 88ms/step - accuracy: 0.8498 - loss: 0.3866 - val_accuracy: 0.6763 - val_loss: 0.6506
Epoch 3/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 81s 86ms/step - accuracy: 0.9632 - loss: 0.1165 - val_accuracy: 0.6763 - val_loss: 1.0991
Epoch 4/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 56s 83ms/step - accuracy: 0.9909 - loss: 0.0340 - val_accuracy: 0.6939 - val_loss: 1.1734
Epoch 5/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 56s 83ms/step - accuracy: 0.9989 - loss: 0.0061 - val_accuracy: 0.7083 - val_loss: 1.7095
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.6698    0.6869    0.6782       313
           1     0.6776    0.6603    0.6688       312

    accuracy                         0.6736       625
   macro avg     0.6737    0.6736    0.6735       625
weighted avg     0.6737    0

# LSTM+BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),  # First LSTM layer (return sequences so next LSTM can take it)
    Bidirectional(LSTM(64, return_sequences=False)),  # BiLSTM on top
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary output
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=4,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/4


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 81s 112ms/step - accuracy: 0.5092 - loss: 0.6905 - val_accuracy: 0.6442 - val_loss: 0.6355
Epoch 2/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 76s 113ms/step - accuracy: 0.8250 - loss: 0.4199 - val_accuracy: 0.6955 - val_loss: 0.6123
Epoch 3/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 74s 110ms/step - accuracy: 0.9627 - loss: 0.1284 - val_accuracy: 0.6907 - val_loss: 0.9150
Epoch 4/4
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 111ms/step - accuracy: 0.9849 - loss: 0.0688 - val_accuracy: 0.6827 - val_loss: 1.4988
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.7022    0.6102    0.6530       313
           1     0.6544    0.7404    0.6947       312

    accuracy                         0.6752       625
   macro avg     0.6783    0.6753    0.6739       625
weighted avg     0.6783    0.6752    0.6738       625



# LSTM+BILSTM+CNN

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# ✅ Load datasets
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")


# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM + CNN model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),
    Bidirectional(LSTM(64, return_sequences=True)),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred = (model.predict(X_test_pad) > 0.5).astype("int32")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/3


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 94s 129ms/step - accuracy: 0.4723 - loss: 0.6984 - val_accuracy: 0.6170 - val_loss: 0.6753
Epoch 2/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 128ms/step - accuracy: 0.6884 - loss: 0.6130 - val_accuracy: 0.7179 - val_loss: 0.6024
Epoch 3/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 140s 125ms/step - accuracy: 0.9229 - loss: 0.2533 - val_accuracy: 0.6955 - val_loss: 0.6506
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.6706    0.7284    0.6983       313
           1     0.7018    0.6410    0.6700       312

    accuracy                         0.6848       625
   macro avg     0.6862    0.6847    0.6842       625
weighted avg     0.6861    0.6848    0.6842       625



# EMOTION

# LSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-val-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights (for balanced learning)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ 4. Build the LSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQ_LEN))
model.add(LSTM(64, return_sequences=False))
#model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dense(6, activation='softmax'))  # 6 classes

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.1713 - loss: 1.7956 - val_accuracy: 0.1026 - val_loss: 1.7951
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.1637 - loss: 1.7931 - val_accuracy: 0.2067 - val_loss: 1.7949
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1708 - loss: 1.7927 - val_accuracy: 0.1026 - val_loss: 1.7923
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.1727 - loss: 1.7919 - val_accuracy: 0.2484 - val_loss: 1.7909
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.1852 - loss: 1.7922 - val_accuracy: 0.1074 - val_loss: 1.7933
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.1674 - loss: 1.7928 - val_accuracy: 0.1923 - val_loss: 1.7919
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.1676 - loss: 1.7916 - val_accuracy: 0.1026 - val_loss: 1.7932
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1517 - loss: 1.7918 - val_accuracy: 0.1426 - va

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# BILSTM

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-val-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ 4. Build the BiLSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQ_LEN))
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dense(6, activation='softmax'))  # 6 classes

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 16s 13ms/step - accuracy: 0.1850 - loss: 1.7675 - val_accuracy: 0.3413 - val_loss: 1.5710
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.5963 - loss: 1.0550 - val_accuracy: 0.3830 - val_loss: 1.4956
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8443 - loss: 0.4149 - val_accuracy: 0.4343 - val_loss: 1.7402
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9555 - loss: 0.1386 - val_accuracy: 0.4872 - val_loss: 2.0296
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9847 - loss: 0.0533 - val_accuracy: 0.4407 - val_loss: 2.3871
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.9945 - loss: 0.0265 - val_accuracy: 0.4615 - val_loss: 2.6615
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9962 - loss: 0.0185 - val_accuracy: 0.4663 - val_loss: 2.6577
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9972 - loss: 0.0124 - va

# LSTM+BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-val-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ 4. Build LSTM + BiLSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQ_LEN))
model.add(LSTM(64, return_sequences=True))  # ✅ Outputs a sequence
model.add(Bidirectional(LSTM(64)))          # ✅ Processes the sequence, outputs a vector
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(6, activation='softmax'))  # 6 emotion classes

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


85/85 ━━━━━━━━━━━━━━━━━━━━ 28s 226ms/step - accuracy: 0.1723 - loss: 1.7924 - val_accuracy: 0.1138 - val_loss: 1.7909
Epoch 2/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 226ms/step - accuracy: 0.2568 - loss: 1.7366 - val_accuracy: 0.3205 - val_loss: 1.5868
Epoch 3/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 207ms/step - accuracy: 0.5710 - loss: 1.1556 - val_accuracy: 0.3910 - val_loss: 1.5485
Epoch 4/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 207ms/step - accuracy: 0.7734 - loss: 0.6572 - val_accuracy: 0.4279 - val_loss: 1.7436
Epoch 5/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 207ms/step - accuracy: 0.8998 - loss: 0.3364 - val_accuracy: 0.4423 - val_loss: 2.1298
Epoch 6/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 20s 233ms/step - accuracy: 0.9393 - loss: 0.2084 - val_accuracy: 0.4087 - val_loss: 2.5749
Epoch 7/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 18s 214ms/step - accuracy: 0.9685 - loss: 0.1314 - val_accuracy: 0.4103 - val_loss: 2.8732
Epoch 8/10
85/85 ━━━━━━━━━━━━━━━━━━━━ 19s 224ms/step - accuracy: 0.9723 - loss: 0.0966 - val_accuracy: 0.400

# LSTM+BILSTM+CNN

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Bidirectional, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/em-train-balanced.csv")
val_df = pd.read_csv("/content/em-val-balanced.csv")
test_df = pd.read_csv("/content/em-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM + CNN model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),
    Bidirectional(LSTM(64, return_sequences=True)),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(6, activation='softmax')  # 🔥 For multi-class classification (6 classes)
])

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 91s 124ms/step - accuracy: 0.1745 - loss: 1.7900 - val_accuracy: 0.3926 - val_loss: 1.6190
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 144s 127ms/step - accuracy: 0.3859 - loss: 1.4300 - val_accuracy: 0.4968 - val_loss: 1.3482
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 140s 124ms/step - accuracy: 0.6890 - loss: 0.7944 - val_accuracy: 0.4984 - val_loss: 1.5407
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 123ms/step - accuracy: 0.8328 - loss: 0.4446 - val_accuracy: 0.4888 - val_loss: 1.7602
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 142s 123ms/step - accuracy: 0.9383 - loss: 0.2117 - val_accuracy: 0.4936 - val_loss: 2.1137
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 142s 123ms/step - accuracy: 0.9690 - loss: 0.1177 - val_accuracy: 0.4567 - val_loss: 2.5940
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 81s 120ms/step - accuracy: 0.9908 - loss: 0.0412 - val_accuracy: 0.3766 - val_loss: 3.1651
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 82s 120ms/step - accuracy: 0.9770 - loss: 0.0858 

# **# VIOLENCE**

# LSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights (for balanced learning)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ 4. Build the LSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQ_LEN))
model.add(LSTM(128, return_sequences=False))
model.add(Dense(128, activation='relu'))
model.add(Dense(3, activation='softmax'))  # 6 classes

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.3105 - loss: 1.1096 - val_accuracy: 0.3141 - val_loss: 1.0991
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.3117 - loss: 1.0907 - val_accuracy: 0.3141 - val_loss: 1.0988
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.2670 - loss: 1.0908 - val_accuracy: 0.3141 - val_loss: 1.0987
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.2364 - loss: 1.1014 - val_accuracy: 0.3429 - val_loss: 1.0986
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5287 - loss: 1.0827 - val_accuracy: 0.3141 - val_loss: 1.0987
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.3227 - loss: 1.0866 - val_accuracy: 0.3141 - val_loss: 1.0988
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.1680 - loss: 1.1018 - val_accuracy: 0.3141 - val_loss: 1.0992
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3233 - loss: 1.1007 - val_accuracy: 0.3429 -

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ 2. Tokenization
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

X_train = tokenizer.texts_to_sequences(train_df['text'])
X_val = tokenizer.texts_to_sequences(val_df['text'])
X_test = tokenizer.texts_to_sequences(test_df['text'])

X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LEN, padding='post')
X_val = pad_sequences(X_val, maxlen=MAX_SEQ_LEN, padding='post')
X_test = pad_sequences(X_test, maxlen=MAX_SEQ_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# ✅ 3. Compute class weights (for balanced learning)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# ✅ 4. Build the BiLSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQ_LEN))
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))  # ✅ 3 classes for violence classification

# ✅ 5. Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ 6. Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 7. Evaluate the model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# ✅ 8. Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 96s 137ms/step - accuracy: 0.4308 - loss: 1.0526 - val_accuracy: 0.6058 - val_loss: 0.8879
Epoch 2/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 136ms/step - accuracy: 0.8081 - loss: 0.5555 - val_accuracy: 0.5897 - val_loss: 1.0171
Epoch 3/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 138s 130ms/step - accuracy: 0.9471 - loss: 0.1623 - val_accuracy: 0.6426 - val_loss: 1.1723
Epoch 4/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 91s 134ms/step - accuracy: 0.9868 - loss: 0.0567 - val_accuracy: 0.6154 - val_loss: 1.4937
Epoch 5/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 141s 133ms/step - accuracy: 0.9920 - loss: 0.0274 - val_accuracy: 0.6010 - val_loss: 1.7511
Epoch 6/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 146s 139ms/step - accuracy: 0.9906 - loss: 0.0266 - val_accuracy: 0.6282 - val_loss: 1.8349
Epoch 7/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 137s 132ms/step - accuracy: 0.9974 - loss: 0.0159 - val_accuracy: 0.6458 - val_loss: 1.8566
Epoch 8/10
675/675 ━━━━━━━━━━━━━━━━━━━━ 88s 131ms/step - accuracy: 0.9934 - loss: 0.0289 

# LSTM+BILSTM

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ 2. Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ 3. Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ 4. Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ 5. Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ 6. Build LSTM + BiLSTM model (no CNN)
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),
    Bidirectional(LSTM(64)),
    Dropout(0.1),
    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(3, activation='softmax')  # 3 classes
])

# ✅ 7. Compile model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# ✅ 8. Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=4,
    class_weight=class_weights,
    verbose=1
)

# ✅ 9. Evaluate model
y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

# ✅ 10. Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 94s 126ms/step - accuracy: 0.4977 - loss: 1.0702 - val_accuracy: 0.5817 - val_loss: 0.9373
Epoch 2/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 140s 123ms/step - accuracy: 0.7870 - loss: 0.6120 - val_accuracy: 0.5994 - val_loss: 0.8657
Epoch 3/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 83s 124ms/step - accuracy: 0.9291 - loss: 0.2673 - val_accuracy: 0.6138 - val_loss: 1.0429
Epoch 4/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 142s 123ms/step - accuracy: 0.9730 - loss: 0.1215 - val_accuracy: 0.6042 - val_loss: 1.3645
Epoch 5/5
675/675 ━━━━━━━━━━━━━━━━━━━━ 142s 124ms/step - accuracy: 0.9780 - loss: 0.0838 - val_accuracy: 0.6154 - val_loss: 1.5757
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.6550    0.5283    0.5849       212
           1     0.5491    0.7123    0.6201       212
           2     0.7374    0.6567    0.6947       201

    accuracy                         0.6320       625
   macro avg     0.6

# LSTM+BILSTM+CNN

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ✅ 1. Load datasets
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
X_train = train_df['text'].astype(str).tolist()
y_train = train_df['label'].tolist()

X_val = val_df['text'].astype(str).tolist()
y_val = val_df['label'].tolist()

X_test = test_df['text'].astype(str).tolist()
y_test = test_df['label'].tolist()

# ✅ Tokenization
vocab_size = 10000
max_length = 100
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# ✅ Convert labels to numpy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

# ✅ Build LSTM + BiLSTM + CNN model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(64, return_sequences=True),
    Bidirectional(LSTM(64, return_sequences=True)),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(3, activation='softmax')  # 🔥 For multi-class classification (3 classes)
])

# ✅ Compile model for multi-class
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ✅ Train model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,
    batch_size=4,
    class_weight=class_weights
)

# ✅ Evaluate on test set
y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))


Epoch 1/3


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


675/675 ━━━━━━━━━━━━━━━━━━━━ 86s 119ms/step - accuracy: 0.3628 - loss: 1.0886 - val_accuracy: 0.6074 - val_loss: 0.8726
Epoch 2/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 81s 117ms/step - accuracy: 0.7457 - loss: 0.6533 - val_accuracy: 0.6554 - val_loss: 0.7909
Epoch 3/3
675/675 ━━━━━━━━━━━━━━━━━━━━ 81s 116ms/step - accuracy: 0.9256 - loss: 0.2476 - val_accuracy: 0.6234 - val_loss: 1.3779
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.6091    0.6321    0.6204       212
           1     0.5507    0.7170    0.6230       212
           2     0.8372    0.5373    0.6545       201

    accuracy                         0.6304       625
   macro avg     0.6657    0.6288    0.6326       625
weighted avg     0.6627    0.6304    0.6322       625

